# RAG dev - parsing the handbook chapters

Scratch notebook for checking `retrieval/parse.py` by eye on the real FAA
Powerplant Handbook chapters. All the logic lives in
`src/industrial_agent/retrieval/`; this only calls it and prints.

## Setup

In [1]:
import os, sys

if not os.path.exists("repo"):
    !git clone -q -b feature/maintenance-docs-rag https://github.com/Cambie26/industrial_agent.git repo
%pip install -q -e "./repo[rag]"

sys.path.insert(0, os.path.abspath("repo/src"))
sys.path.insert(0, os.path.abspath("repo"))

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for industrial-agent (pyproject.toml) ... done


## Corpus

Upload one PDF per chapter to `repo/data/raw/`, named with the chapter number
(`ch10.pdf`). `data/` is gitignored, so they won't be committed. To read them
from Drive instead, mount it and point `RAW_DIR` at the folder.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

RAW_DIR = Path("drive/MyDrive/faa_powerplant")

#RAW_DIR = Path("repo/data/raw")

In [3]:
RAW_DIR

PosixPath('drive/MyDrive/faa_powerplant')

In [5]:
assert RAW_DIR.is_dir(), f"Missing folder: {RAW_DIR}"
print(sorted(p.name for p in RAW_DIR.glob("*.pdf")))

['ch10.pdf']


In [ ]:
#!cd repo && python scripts/build_index.py --raw-dir /content/drive/MyDrive/faa_powerplant --index-dir /content/drive/MyDrive/faa_powerplant/index

ch10.pdf: 869 blocks -> 201 chunks
modules.json: 100% 349/349 [00:00<00:00, 1.02MB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 469kB/s]
README.md: 100% 94.8k/94.8k [00:00<00:00, 72.6MB/s]
sentence_bert_config.json: 100% 52.0/52.0 [00:00<00:00, 212kB/s]
config.json: 100% 743/743 [00:00<00:00, 4.52MB/s]

model.safetensors: downloading bytes:  49% 64.9M/133M [00:01<00:00, 97.3MB/s, 4.33MB/s  ]
model.safetensors: reconstructing file:  50% 67.0M/133M [00:01<00:01, 58.8MB/s]
model.safetensors: downloading bytes: 100% 86.7M/86.7M [00:01<00:00, 54.0MB/s, 8.10MB/s  ]
model.safetensors: reconstructing file: 100% 133M/133M [00:01<00:00, 83.1MB/s, 12.7MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 7175.11it/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 2.11MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 57.9MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 120MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 450kB/s]
config.json: 100% 190/190 [00:00<00:00, 8

In [ ]:
from collections import Counter

from industrial_agent.retrieval import parse


pdf_paths = parse.chapter_pdfs(RAW_DIR)
for pdf_path in pdf_paths:
    print(f"chapter {parse.chapter_number(pdf_path):>2}: {pdf_path.name}")

chapter 10: ch10.pdf


## Parse one chapter

In [ ]:
CHAPTER = 10
pdf_path = next(path for path in pdf_paths if parse.chapter_number(path) == CHAPTER)

blocks = parse.parse_pdf(pdf_path)
pages = {block.page for block in blocks}
print(f"{pdf_path.name}: {len(pages)} pages, {len(blocks)} blocks")
print(Counter(block.kind for block in blocks))

ch10.pdf: 65 pages, 869 blocks
Counter({'text': 648, 'heading': 132, 'caption': 83, 'table': 6})


### Contents

Chunking takes its section boundaries from this. If it raises, the chapter
PDF has no bookmarks and the contents page will need parsing instead.

In [ ]:
import inspect, dataclasses
print(inspect.getsource(parse.parse_pdf))
print(type(blocks), type(blocks[0]))
print(blocks[0])
if dataclasses.is_dataclass(blocks[0]):
    print([f.name for f in dataclasses.fields(blocks[0])])

def parse_pdf(pdf_path: Path | str, table_aware: bool = True) -> list[Block]:
    """Parse one chapter PDF into blocks, page by page.

    With `table_aware=False` each page is one text block from PyMuPDF's plain
    text extraction - the naive baseline for the ablation.
    """
    import pymupdf

    pdf_path = Path(pdf_path)
    if not pdf_path.exists():
        raise FileNotFoundError(f"No PDF at {pdf_path}.")
    chapter = chapter_number(pdf_path)

    blocks: list[Block] = []
    with pymupdf.open(pdf_path) as document:
        # Loop through each page, numbering from 1 to match the printed PDF.
        for page_index, page in enumerate(document):
            page_number = page_index + 1
            if table_aware:
                blocks.extend(parse_page(page, pdf_path.name, chapter, page_number))
                continue
            text = page.get_text("text").strip()
            if text:
                blocks.append(Block(pdf_path.name, chapter, page_number, "text", text))



In [ ]:
for level, title, page in parse.table_of_contents(pdf_path):
    print(f"{'  ' * (level - 1)}{title}  (p.{page})")

Structure Bookmarks  (p.-1)
  Reciprocating Engine Overhaul  (p.1)
    Reciprocating Engine Overhaul  (p.1)
      Reciprocating Engine Overhaul  (p.1)
        Reciprocating Engine Overhaul  (p.1)
        Both maintenance and complete engine overhauls are performed normally at specified intervals. This interval is usually governed by the number of hours the powerplant has been in operation. The actual overhaul period for a specific engine is generally determined by the manufacturer’s recommendations. Each engine manufacturer sets a total time in service when the engine should be removed from service and overhauled. Depending upon how the engine is used in service, the overhaul time can be mandatory. The over  (p.1)
        A complete overhaul process includes the following ten steps: receiving inspection; disassembly; visual inspection; cleaning; structural inspection; non-destructive testing (NDT) inspection; dimensional inspection; repair and replacement; reassembly; and testing and b

### Reading order on one page

Check the two columns come out left then right, not interleaved.

In [ ]:
PAGE = 3
for block in blocks:
    if block.page == PAGE:
        print(f"[{block.kind}] {block.text[:200]}\n")

[heading] Visual Inspection

[text] Visual inspection should precede all other inspection procedures. Parts should not be cleaned before a preliminary visual inspection, since indications of a failure may often be detected from the resi

[text] Several terms are used to describe defects detected in engine parts during inspection. Some of the more common terms and definitions are:

[text] 1. Abrasion—an area of roughened scratches or marks usually caused by foreign matter between moving parts or surfaces.

[text] 2. Brinelling—one or more indentations on bearing races, usually caused by high static loads or application of force during installation or removal. Indentations are rounded or spherical due to the im

[text] 3. Burning—surface damage due to excessive heat. It is usually caused by improper fit, defective lubrication, or over-temperature operation.

[text] 4. Burnishing—polishing of one surface by sliding contact with a smooth, harder surface. Usually no displacement nor removal

### Tables, as markdown

In [ ]:
from IPython.display import Markdown, display

tables = [block for block in blocks if block.kind == "table"]
print(f"{len(tables)} tables")
for table in tables[:5]:
    display(Markdown(f"**page {table.page}**\n\n{table.text}"))

6 tables


**page 36**

| Trouble | Probable causes | Remedy |
| --- | --- | --- |
| Engine fails to start. | • Lack of fuel. | • Check fuel system for leaks fill fuel tank. • Clean dirty lines, strainers, or fuel valves. |
|  | • Underpriming. | • Use correct priming procedure. |
|  | • Overpriming. | • Open throttle and “unload” engine by rotating the propeller. |
|  | • Incorrect throttle setting. | • Open throttle to one-tenth of its range. |
|  | • Defective spark plugs. | • Clean and re-gap or replace spark plugs. |
|  | • Defective ignition wire. | • Test and replace any defective wires. |
|  | • Defective or weak battery. | • Replace with charged battery. |
|  | • Improper operation of magneto or breaker points. | • Check internal timing of magnetos. |
|  | • Water in carburetor. | • Drain carburetor and fuel lines. |
|  | • Internal failure. | • Check oil sump strainer for metal particles. |
|  | • Magnetized impulse coupling, if installed. | • Demagnetize impulse coupling. |
|  | • Frozen spark plug electrodes. | • Replace spark plugs or dry out plugs. |
|  | • Mixture control in idle cutoff. | • Open mixture control. |
|  | • Shorted ignition switch or loose ground. | • Check and replace or obtain correct idle. |
| Engine fails to idle properly. | • Incorrect carburetor idle speed adjustment. | • Adjust throttle stop to obtain correct idle. |
|  | • Incorrect idle mixture. | • Adjust mixture. (Refer to engine manufacturer’s handbook for proper procedure.) |
|  | • Leak in the induction system. | • Tighten all connections in the induction system. Replace any defective parts. |
|  | • Low cylinder compression. | • Check cylinder compression. |
|  | • Faulty ignition system. | • Check entire ignition system. |
|  | • Open or leaking primer. | • Lock or repair primer. |
|  | • Improper spark plug setting for altitude. | • Check spark plug gap. |
|  | • Dirty air filter. | • Clean or replace. |
| Low power and engine running uneven. Troubleshooting op | • Mixture too rich; indicated by sluggish engine operation, red exhaust flame, and black smoke. | • Check primer. Re-adjust carburetor mixture. |
|  | • Mixture too lean; indicated by overheating or backfiring. | • Check fuel lines for dirt or other restrictions. Check fuel supply. |
|  | • Leaks in induction system. | • Tighten all connections. Replace defective parts. |
|  | • Defective spark plugs. | • Clean or replace spark plugs. |
|  | • Improper grade of fuel. | • Fill tank with recommended grade. |
|  | • Magneto breaker points not working properly. | • Clean points. Check internal timing of magneto. |
|  | • Defective ignition wire. | • Test and replace any defective wires. |
|  | • Defective spark plug terminal connectors. | • Replace connectors on spark plug wire. |
|  | • Incorrect valve clearance. | • Adjust valve clearance. |
|  |  | • Check and replace or repair. |
|  | pposed engines. |  |

**page 37**

| Trouble |  | Probable causes | Remedy |
| --- | --- | --- | --- |
|  | Low power and engine running uneven. | • Restriction in exhaust system. | • Remove restriction. |
|  |  | • Improper ignition timing. | • Check magnetos for timing and synchronization. |
|  | Engine fails to develop full power. | • Throttle lever out of adjustment. | • Adjust throttle lever. |
|  |  | • Leak in induction system. | • Tighten all connections and replace defective parts. |
|  |  | • Restriction in carburetor airscoop. | • Examine airscoop and remove restriction. |
|  |  | • Improper fuel. | • Fill tank with recommended fuel. |
|  |  | • Propeller governor out of adjustment. | • Adjust governor. |
|  |  | • Faulty ignition. | • Tighten all connections. Check system. Check ignition timing. |
|  | Rough running engine. | • Cracked engine mount(s). | • Repair or replace engine mount(s). |
|  |  | • Unbalanced propeller. | • Remove propeller and have it checked for balance. |
|  |  | • Defective mounting bushings. | • Install new mounting bushings. |
|  |  | • Lead deposit on spark plugs. | • Clean or replace plugs. |
|  |  | • Primer unlocked. | • Lock primer. |
|  | Low oil pressure. | • Insufficient oil. | • Check oil supply. |
|  |  | • Dirty oil strainers. | • Remove and clean oil strainers. |
|  |  | • Defective pressure gauge. | • Replace gauge. |
|  |  | • Air lock or dirt in relief valve. | • Remove and clean oil pressure relief valve. |
|  |  | • Leak in suction line or pressure line. | • Check gasket between accessory housing crankcase. |
|  |  | • High oil temperature. | • See “high oil temperature” in trouble column. |
|  |  | • Stoppage in oil pump intake passage. | • Check line for obstruction. Clean suction strainer. |
|  |  | • Worn or scored bearings. | • Overhaul engine. |
|  | High oil temperature. | • Insufficient air cooling. | • Check air inlet and outlet for deformation or obstruction. |
|  |  | • Insufficient oil supply. | • Fill oil tank to proper level. |
|  |  | • Clogged oil lines or strainers. | • Remove and clean oil line or strainers. |
|  |  | • Failing or failed bearings. | • Examine sump for metal particles and, if found, overhaul engine. |
|  |  | • Defective thermostats. | • Replace thermostats. |
|  |  | • Defective temperature gauge. | • Usually caused by weak or stuck rings. |
|  |  | • Excessive blow-by. | • Overhaul engine. |
| Excessive oil consumption. |  | • Failing or failed bearing. | • Check sump for metal particles and if found, an overhaul of engine is indicated. |
|  |  | • Worn or broken piston rings. | • Install new rings. |
|  |  | • Incorrect installation of piston rings. | • Install new rings. • Check engine carefully for leaking gaskets or |
|  |  | • External oil leakage. | • Check engine carefully for leaking gaskets or O-rings. |
|  |  | • Leakage through engine fuel pump vent. | • Replace fuel pump seal. |
|  |  | • Engine breather or vacuum pump breather. | • Check engine, and overhaul or replace vacuum pump. |

**page 44**

| Cylinder No. | Temperature readings |  |
| --- | --- | --- |
|  | Right magneto | Left magneto |
| 1 | 180 | 170 |
| 2 | 170 | 175 |
| 3 | 100 | 170 |
| 4 | 145 | 60 |
| 5 | 70 | 155 |
| 6 | 60 | 45 |

**page 60**

| Indicated Malfunction |  | Probable Causes | Suggested Action |
| --- | --- | --- | --- |
|  | Engine has low rpm, exhaust gas temperature, and fuel flow when set to expected engine pressure ratio. | • Engine pressure ratio indication has high reading error. | • Check inlet pressure line from probe to transmitter for leaks. |
|  |  |  | • Check engine pressure ratio transmitter and indicator for accuracy. |
|  | Engine has high rpm, exhaust gas temperature, and fuel flow when set to expect engine pressure ratio. | • Engine pressure ratio indication has low reading error due to: |  |
|  |  | - Misaligned or cracked turbine discharge probe. | • Check probe condition. |
|  |  | - Leak in turbine discharge pressure line from probe to transmitter. | • Pressure-test turbine discharge pressure line for leaks. |
|  |  | - Inaccurate engine pressure ratio transmitter or indicator. - Carbon particles collected in turbine discharge pressure line or restrictor orifices. | • Check engine pressure ratio transmitter and indicator for accuracy. |
|  | Engine has high exhaust gas temperature, low rpm, and high fuel flow at all engine pressure ratio settings. | • Possible turbine damage and/or loss of turbine efficiency. | • Confirm indication of turbine damage by: - Checking engine coast-down for abnormal noise and reduced time. - Visually inspect turbine area with strong light. |
|  | Note: Engines with damage in turbine section may have tendency to hang up during starting. | • If only exhaust gas temperature is high, other parameters normal, the problem may be thermocouple leads or instrument. | • Re-calibrate exhaust gas temperature instrumentation. |
|  | Engine vibrates throughout rpm range, but indicated amplitude reduces as rpm is reduced. | • Turbine damage. | • Check turbine as outlined in preceding item. |
|  | Engine vibrates at high rpm and fuel flow when compared to constant engine pressure ratio. | • Damage in compressor section. | • Check compressor section for damage. |
|  | Engine vibrates throughout rpm range, but is more pronounced in cruise or idle rpm range. | • Engine-mounted accessory such as constant-speed drive, generator, hydraulic pump, etc. | • Check each component in turn. |
|  | No change in power setting parameters, but oil temperature high. | • Engine main bearings. | • Check scavenge oil filters and magnetic plugs. |
|  | Engine has higher than normal exhaust gas temperature during takeoff, climb, and cruise. Rpm and fuel flow higher than normal. | • Engine bleed-air valve malfunction. | • Check operation of bleed valve. |
|  |  | • Turbine discharge pressure probe or line to transmitter leaking. | • Check condition of probe and pressure line to transmitter. |
| Engine has high exhaust gas temperature at target engine pressure ratio for takeoff. Troubleshooting tur |  | • Engine out of trim. | • Check engine with jetcal. Re-trim as desired. |
|  |  | rbojet engines. |  |

**page 61**

| Indicated Malfunction |  | Probable Causes | Suggested Action |
| --- | --- | --- | --- |
|  | Engine rumbles during starting and at low power cruise conditions. | • Pressurizing and drain valve malfunction. | • Replace pressurizing and drain valves. |
|  |  | • Cracked air duct. | • Repair or replace duct. |
|  |  | • Fuel control malfunction. | • Replace fuel control. |
|  | Engine rpm hangs up during starting. | • Subzero ambient temperatures. | • If hang-up is due to low ambient temperature, engine usually can be started by turning on fuel booster pump or by positioning start lever to run earlier in the starting cycle. |
|  |  | • Compressor section damage. | • Check compressor for damage. |
|  |  | • Turbine section damage. | • Inspect turbine for damage. |
|  | High oil temperature. | • Scavenge pump failure. | • Check lubricating system and scavenge pumps. |
|  |  | • Fuel heater malfunction. | • Replace fuel heater. |
|  | High oil consumption. | • Scavenge pump failure. | • Check scavenge pumps. |
|  |  | • High sump pressure. | • Check sump pressure as outlined in |
|  |  | • Gearbox seal leakage. | manufacturer’s maintenance manual. • Check gearbox seal by pressurizing overboard vent. |
| Overboard oil loss. |  | • Can be caused by high airflow through the tank, foaming oil, or unusual amounts of oil returned to the tank through the vent system. | • Check oil for foaming. • Vacuum-check sumps. • Check scavenge pumps. |

### Figure captions

In [ ]:
for caption in (block for block in blocks if block.kind == "caption"):
    print(f"p.{caption.page:>3}  {caption.text}")

p.  2  Figure 10-1. Engine mounted on an overhaul stand.
p.  4  Figure 10-2. Checking a piston head for flatness.
p.  5  Figure 10-3. Typical solvent degreasing tank.
p.  6  Figure 10-4. Grit-blasting machine.
p.  8  Figure 10-5. A cylinder bore gauge.
p.  9  Figure 10-6. Ridge or step formed in an engine cylinder.
p.  9  Figure 10-7. Checking crankshaft runout.
p.  9  Figure 10-8. A micrometer.
p. 10  Figure 10-9. Telescoping gauges and micrometer combination.
p. 10  Figure 10-10. Checking connecting rod squareness.
p. 11  Figure 10-11. Checking a piston pin for bends.
p. 12  Figure 10-12. Valve showing locations for checking runout and section for measuring edge thickness.
p. 12  Figure 10-13. Valve face surface.
p. 12  Figure 10-14. Checking valve stretch with a manufacturer’s gauge.
p. 12  Figure 10-15. Valve spring compression tester.
p. 13  Figure 10-16. Valve seat grinding equipment.
p. 13  Figure 10-17. Valve seat grinding setup.
p. 14  Figure 10-18. Valve seat grinding.
p. 15 

## Naive vs table-aware

The naive parse is the ablation baseline: plain page text, tables flattened
into the prose. Compare the same page both ways.

In [ ]:
naive_blocks = parse.parse_pdf(pdf_path, table_aware=False)
table_page = tables[0].page if tables else PAGE

print("=== naive ===")
print(next(block.text for block in naive_blocks if block.page == table_page)[:1500])
print("\n=== table-aware ===")
print(parse.to_markdown([block for block in blocks if block.page == table_page])[:1500])

=== naive ===
10-36
• Lack of fuel.
• Underpriming.
• Overpriming.
• Incorrect throttle setting.
• Defective spark plugs.
• Defective ignition wire.
• Defective or weak battery.
• Improper operation of magneto or breaker 
points.
• Water in carburetor.
• Internal failure.
• Magnetized impulse coupling, if installed.
• Frozen spark plug electrodes.
• Mixture control in idle cutoff.
• Shorted ignition switch or loose ground.
• Incorrect carburetor idle speed adjustment.
• Incorrect idle mixture.
• Leak in the induction system.
• Low cylinder compression.
• Faulty ignition system.
• Open or leaking primer.
• Improper spark plug setting for altitude.
• Dirty air filter.
• Mixture too rich; indicated by sluggish engine
  operation, red exhaust flame, and black smoke.
• Mixture too lean; indicated by overheating or
  backfiring.
• Leaks in induction system.
• Defective spark plugs.
• Improper grade of fuel.
• Magneto breaker points not working properly.
• Defective ignition wire.
• Defective

## Save the parse for reading in full

In [ ]:
PARSED_DIR = Path("repo/data/parsed")
PARSED_DIR.mkdir(parents=True, exist_ok=True)

markdown_path = PARSED_DIR / f"{pdf_path.stem}.md"
markdown_path.write_text(parse.to_markdown(blocks))
print(f"wrote {markdown_path} ({markdown_path.stat().st_size / 1024:.0f} kB)")

wrote repo/data/parsed/ch10.md (230 kB)


## Whole corpus

In [ ]:
corpus_blocks = parse.parse_corpus(RAW_DIR)
by_chapter = Counter((block.chapter, block.kind) for block in corpus_blocks)
for chapter in sorted({block.chapter for block in corpus_blocks}):
    counts = {kind: by_chapter[(chapter, kind)] for kind in ("text", "table", "caption")}
    print(f"chapter {chapter:>2}: {counts}")

chapter 10: {'text': 648, 'table': 6, 'caption': 83}


# Testing

In [4]:
from industrial_agent.retrieval.index import load_index
from industrial_agent.retrieval.search import search

index = load_index("/content/drive/MyDrive/faa_powerplant/index")

def ask(question, turbine_only=True):
    for hit in search(question, index=index, turbine_only=turbine_only):
        print(f"{hit.score:.2f}  {hit.chunk.citation}  {hit.chunk.section} > {hit.chunk.subsection}")
        print("      ", hit.chunk.text.split("\n\n", 1)[1][:200], "\n")

#ask("What are the best practices when inspecting turbine blades?")

In [5]:
Q = "How do I gap a spark plug?"
print(f"--- Q: {Q} ---")
for hit in search(f"{Q}", index=index):
    print(f"{hit.score:.2f}  {hit.chunk.citation}  {hit.chunk.text[:]!r}")

--- Q: How do I gap a spark plug? ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

0.00  Ch. 10, pp. 10-49-50  'Combustion Section Inspection > Inspection & Repair of Combustion Chambers\n\nInspect the combustion chambers and covers for cracks by using visible dye or fluorescent penetrant inspection method. Any cracks, nicks, or dents are usually cause for rejecting the component. Inspect the covers, noting particularly the area around the fuel drain bosses for any pits or corrosion. When repairing the combustion chamber liner, the procedures given in the appropriate engine manufacturer’s overhaul instruction manual should be followed. If there is doubt that the liner is serviceable, it should be replaced.\n\nCombustion chambers should be replaced or repaired if two cracks are progressing from a free edge so that their meeting is imminent and could allow a piece of metal that could cause turbine damage to break loose. Separate cracks in the baffle are acceptable. Cracks in the cone are rare but, at any location on this component, is cause for rejection of the liner. 

In [6]:
Q = "What is the torque for the CFM56 fan blade bolts?"
print(f"--- Q: {Q} ---")
for hit in search(f"{Q}", index=index):
    print(f"{hit.score:.2f}  {hit.chunk.citation}  {hit.chunk.text[:]!r}")

--- Q: What is the torque for the CFM56 fan blade bolts? ---
0.00  Ch. 10, p. 10-45  'Turbine Engine Maintenance > Causes of Blade Damage\n\nLoose objects often enter an engine either accidentally or through carelessness. Foreign object damage (FOD), such as pencils, tools, and flashlights, are often drawn into the engine and can cause damage to the fan blades. [Figure 10-59] Do not carry any objects in pockets when working around operational turbine engines.\n\nA compressor rotor can be damaged beyond repair by tools that are left in the air intake, where they are drawn into the engine on subsequent starts. A simple solution to the problem is to check the tools against a tool checklist. Prior to starting a turbine engine, make a minute inspection of engine inlet ducts to assure that items, such as nuts, bolts, lock wire, or tools, were not left there after work had been performed.\n\nFigure 10-60 shows some examples of blade damage to an axial-flow engine. The descriptions and possibl

In [7]:
for Q, turbine_only in [("Engine has high EGT at takeoff power. What should I check?", True),
                        ("How do I gap a spark plug?", False)]:
    print(f"--- Q: {Q}  (turbine_only={turbine_only}) ---")
    for hit in search(Q, index=index, turbine_only=turbine_only):
        print(f"{hit.score:.3f}  {hit.chunk.citation}  {hit.chunk.engine_type}  {hit.chunk.section}")

--- Q: Engine has high EGT at takeoff power. What should I check?  (turbine_only=True) ---
0.739  Ch. 10, p. 10-54  turbine  Engine Ratings
0.579  Ch. 10, p. 10-60  turbine  Turbine Engine Calibration & Testing
0.402  Ch. 10, p. 10-59  turbine  Turbine Engine Calibration & Testing
0.211  Ch. 10, pp. 10-59-61  turbine  Turbine Engine Calibration & Testing
0.146  Ch. 10, p. 10-55  turbine  Turbine Engine Instruments
--- Q: How do I gap a spark plug?  (turbine_only=False) ---
0.260  Ch. 10, p. 10-36  reciprocating  Engine Troubleshooting
0.160  Ch. 10, p. 10-36  reciprocating  Engine Troubleshooting
0.045  Ch. 10, p. 10-27  reciprocating  Reciprocating Engine Operation
0.011  Ch. 10, p. 10-44  reciprocating  Cold Cylinder Check
0.010  Ch. 10, p. 10-13  reciprocating  Dimensional Inspection
